In [ ]:
# simple event loop

from queue import Queue
from functools import partial

eventloop = None

class EventLoop(Queue):
    def start(self):
        num = 1
        while num <=6:
            function = self.get()
            function()
            num += 1

def do_hello():
    global eventloop
    print("hello")
    eventloop.put(do_world)

def do_world():
    global eventloop
    print("world")
    eventloop.put(do_hello)


eventloop = EventLoop()
eventloop.put(do_hello)
eventloop.start()

hello
world
hello
world
hello
world


In [ ]:
import asyncio

# 定义一个协程
async def fetch_data():
    print("1. 开始请求网络...")
    # 模拟网络 I/O：遇到 await，当前协程暂停，交出控制权！
    await asyncio.sleep(2) 
    print("2. 网络请求完成，继续执行...")
    return "数据"

# 调用协程函数，会返回一个协程对象（不会立刻执行）
coro = fetch_data()

# 要让它跑起来，必须扔进 Event Loop 里
# 不能使用 asyncio.run(coro)
await coro

1. 开始请求网络...
2. 网络请求完成，继续执行...


'数据'

> asyncio.run()：从同步世界进入异步世界的入口；

> await：在异步世界内部，等待另一个异步操作完成，并把控制权交还给 Event Loop。


使用时间循环的编程方式有两大类： 回调和期票

回调： 指明callback的参数，函数不返回其结果；而是调用callback指定的函数，并将结果作为参数。

In [ ]:
import asyncio

async def work1():
    await asyncio.sleep(2)
    print("work 1 done")

async def work2():
    await asyncio.sleep(1)
    print("work 2 done")

# 串行：总耗时约 3 秒
# 连续 await a(); await b() 是串行：等 a 完再 b。
async def main():
    await work1()
    await work2()

await main()

work 1 done
work 2 done


In [ ]:
# 并发：总耗时约 2 秒
# 先 create_task(a())、create_task(b()) 再分别 await，才是并发
async def main():
    t1 = asyncio.create_task(work1())
    t2 = asyncio.create_task(work2())
    await t1
    await t2

await main()

work 2 done
work 1 done


async 是“不阻塞等待”的代码写法；并发是“多个任务交错推进”的执行效果。async/await 可以用来实现并发，但单独 await 是串行；并发也可以用线程、进程、回调实现，不一定要 async。